# Tokens, and why you pay per token

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 8 §8.2 · type: concept-lab

**The promise:** by the end you can count tokens with the *right* tokenizer, and explain a model's letter-counting and multilingual-cost quirks from how text gets chunked.

This is the first runnable notebook in Part III. It runs fully offline and free — `tiktoken` is a local library, no API key needed.

## 🧠 Why this matters

A language model never sees your characters or your words. It sees **tokens** — chunks produced by a tokenizer that was trained to compress common text efficiently. In English prose a token averages roughly four characters, so `"unbelievable"` might split into `un` + `believ` + `able`, while `"the"` is a single token.

This is not trivia. Tokens are the unit of *everything you pay for and wait for*: API pricing is per token, context limits are in tokens, latency scales with tokens. If you can't count them, you can't budget cost or stay under a context limit. And a whole family of "why is the model so dumb?" behaviors — miscounting letters, fumbling long arithmetic, costing 3× more in other languages — falls straight out of how text is chunked. See §8.2.

## Objectives & prereqs

**By the end you can:**
- Encode text into token ids with `tiktoken` and read the `~4-chars-per-token` rule of thumb.
- Predict and then measure the multilingual cost multiplier.
- Explain the "count the r's in strawberry" failure from tokenization alone.
- Reach for `client.messages.count_tokens(...)` when you need an *exact* Anthropic count.

**Prereqs:** Chapter 8 read. No prior notebook required — this is the entry point to Part III.

**Packages:** `tiktoken` (in `requirements.txt`) plus the standard library. `anthropic` is only touched on the live path.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default) keeps everything offline, free, and deterministic.
# MOCK=0 enables the live Anthropic token-count call near the end (a few free-tier
# count calls; counting is not billed as generation).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

import tiktoken  # local tokenizer library — no network, no key

# o200k_base is the tokenizer family used by recent OpenAI models. It is a good,
# concrete stand-in for *seeing* how byte-pair tokenization chunks text. Exact
# Anthropic counts come from the counting endpoint (shown later), not a local approx.
enc = tiktoken.get_encoding("o200k_base")

print(f"MOCK mode: {MOCK}  (offline, free)" if MOCK else "LIVE mode: will call the Anthropic count endpoint")
print(f"tokenizer: o200k_base, vocab size = {enc.n_vocab:,}")

## The model reads chunks, not words

Let's encode a few strings and look at the actual token ids — and, crucially, decode each id back to the text fragment it stands for. That decoded fragment is what the model actually "sees."

In [ ]:
def show(text: str) -> None:
    """Encode a string and print its tokens as the fragments the model sees."""
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]
    chars_per_token = len(text) / len(ids) if ids else 0.0
    print(f"{text!r}")
    print(f"  {len(ids)} tokens, {len(text)} chars  (~{chars_per_token:.2f} chars/token)")
    print(f"  ids:    {ids[:12]}{' ...' if len(ids) > 12 else ''}")
    print(f"  pieces: {pieces[:12]}{' ...' if len(pieces) > 12 else ''}")
    print()


show("the")
show("unbelievable")
show("The agent retried the API call after a 429 response.")

**What you just saw.** `"the"` is one token; `"unbelievable"` fractures into a handful of sub-word pieces; the 429-error sentence lands close to the `~4 chars/token` rule. The model's entire view of your text is that `pieces` list — it has no access to the letters inside a chunk.

## 🔮 Predict: English vs. non-English

Here is the same idea ("The agent retried the request.") in English and in three other scripts. Before you run the next cell, **predict**: roughly how many *more* tokens will the non-English versions cost per sentence? 1.2×? 2×? 4×?

Write your guess down, then run it.

In [ ]:
sentences = {
    "English":  "The agent retried the request.",
    "Spanish":  "El agente reintent\u00f3 la solicitud.",
    "Japanese": "\u30a8\u30fc\u30b8\u30a7\u30f3\u30c8\u306f\u30ea\u30af\u30a8\u30b9\u30c8\u3092\u518d\u8a66\u884c\u3057\u307e\u3057\u305f\u3002",
    "Hindi":    "\u090f\u091c\u0947\u0902\u091f \u0928\u0947 \u0905\u0928\u0941\u0930\u094b\u0927 \u0915\u093e \u092a\u0941\u0928\u0903 \u092a\u094d\u0930\u092f\u093e\u0938 \u0915\u093f\u092f\u093e\u0964",
}

baseline = len(enc.encode(sentences["English"]))
for lang, text in sentences.items():
    n = len(enc.encode(text))
    mult = n / baseline
    print(f"{lang:9s} {n:3d} tokens   ({mult:.1f}x English)")

**What you just saw.** The same meaning can cost several times more tokens in another language, because the tokenizer's vocabulary was optimized mostly on English-heavy text. For a multilingual product this silently multiplies cost *and* eats context budget — a "4,000-token" prompt in English can be far larger translated.

## Why "count the r's in strawberry" is hard

Models are famously shaky at counting letters or reversing strings. It looks like stupidity; it's tokenization. The model never sees letters — only token chunks — so a question *about the letters* is asking it to introspect inside a unit it can't open.

In [ ]:
word = "strawberry"
ids = enc.encode(word)
pieces = [enc.decode([i]) for i in ids]

print(f"{word!r} -> {len(ids)} token(s): {pieces}")
print(f"actual letter count of 'r': {word.count('r')}")
print()
print("The model sees the chunk(s) above, not 's-t-r-a-w-b-e-r-r-y'. To answer")
print("'how many r's', it would have to reconstruct the spelling from chunks it")
print("only ever learned as opaque units. Same root cause as reversing a string")
print("or doing digit-by-digit math on a long number.")

In [ ]:
# Long numbers get grouped into arbitrary token chunks too — which is why
# long-multiplication-style arithmetic is unreliable.
for number in ["7", "42", "1234567", "900000000001"]:
    ids = enc.encode(number)
    print(f"{number:>13s} -> {len(ids)} token(s): {[enc.decode([i]) for i in ids]}")

**What you just saw.** Digits don't map one-token-per-digit; they clump in ways the tokenizer found efficient. The model is doing arithmetic over those clumps, not over digits, so errors on long numbers are expected, not surprising.

## Exact Anthropic counts: the counting endpoint

`tiktoken` is great for *seeing* how chunking works, but every provider tokenizes differently. When you need the exact token count for a model you're actually calling, use that provider's counting method. For Anthropic models that is `client.messages.count_tokens(...)` — it returns the real input-token count for the request, and counting is not billed like generation.

Below, `MOCK=1` returns a canned, realistic count so the notebook stays offline. Flip `COMPANION_MOCK=0` (with `ANTHROPIC_API_KEY` set) to hit the live endpoint.

In [ ]:
MODEL = "claude-opus-4-8"  # the book's default model
prompt = "The agent retried the API call after a 429 response."


def anthropic_token_count(text: str) -> int:
    """Exact Anthropic input-token count for `text`.

    MOCK=1 returns a canned value so this runs offline; MOCK=0 calls the real
    counting endpoint. Secrets are read from the environment only.
    """
    if MOCK:
        # A realistic canned count for the prompt above on a recent Claude model.
        # (Anthropic's tokenizer differs from o200k_base, so this need not match
        # the local tiktoken count — that is exactly the point.)
        return 15

    import anthropic  # imported lazily so MOCK users don't need the SDK configured

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    resp = client.messages.count_tokens(
        model=MODEL,
        messages=[{"role": "user", "content": text}],
    )
    return resp.input_tokens


local_approx = len(enc.encode(prompt))
exact = anthropic_token_count(prompt)
print(f"prompt: {prompt!r}")
print(f"  tiktoken (o200k_base) local approx : {local_approx} tokens")
print(f"  Anthropic count_tokens ({'MOCK' if MOCK else 'live'})       : {exact} tokens")
print()
print("Different tokenizers, different counts. Budget with the one you'll call.")

## ⚠️ Pitfall: estimating tokens the wrong way

Two production-breaking habits, both rooted in this notebook:

1. **Estimating tokens by word or character count.** "About 750 words, so ~1,000 tokens" is a guess that fails badly on code, numbers, and non-English text — the very inputs where you most need to be right.
2. **Budgeting with one provider's tokenizer while calling another's model.** A prompt measured as "4,000 tokens" with `tiktoken` can overflow a real Anthropic context limit or blow your cost forecast, because the two tokenizers chunk differently.

**The fix:** count with the *same* tokenizer (or counting endpoint) as the model you are calling. Treat token count as a measured quantity, not an eyeballed one.

## 🎯 Senior lens

A senior engineer wires token counting into the system, not into a one-off script. Cost dashboards, context-budget checks before a call, and truncation logic all read from a single counting helper that uses the *real* tokenizer for the target model — so when you swap models (Chapter 11 makes the model a swappable component), the budgeting follows automatically. They also treat multilingual inputs as a first-class cost line, not an afterthought: a feature that is cheap in English can be 3× the bill in another market, and that shows up in the unit economics, not in a surprise invoice.

## Recap

- Models read **tokens** (sub-word chunks), never characters or words; English averages ~4 chars/token.
- Tokens are the unit of **cost, context, and latency** — so you must be able to count them.
- Non-English text often costs **several times more tokens** per unit of meaning.
- Letter-counting, string-reversal, and long-number arithmetic fail because the model only sees chunks.
- For exact Anthropic counts, use `client.messages.count_tokens(...)`, not a local approximation.

## Exercises

Each exercise *changes* something and asks you to predict the result before running.

1. **Code vs. prose.** Encode a 5-line Python function and an English paragraph of similar character length. Predict which has more tokens, then measure the chars/token for each. What does that imply for the cost of code-heavy prompts?
2. **Your own language.** Add a sentence in a language you read to the multilingual cell and compute its multiplier vs. English. Was it closer to 1.2× or 4×? Why might that be?
3. **Budget check.** Write a `fits_in_context(text, limit)` helper that returns `True`/`False` using `anthropic_token_count`. Predict whether a 2,000-word document fits in a 4,000-token budget, then check.
4. **(Live, optional)** With `COMPANION_MOCK=0` and a key set, count the same prompt with both `tiktoken` and the Anthropic endpoint. How far apart are they?

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Next notebook:** [`08-02-meaning-is-geometry.ipynb`](08-02-meaning-is-geometry.ipynb) — once text is tokens, what are those tokens *for*? Embeddings turn text into geometry, and similar meanings land near each other.
- **Book:** revisit §8.2 (tokens) and look ahead to §8.5 (context windows, also measured in tokens).
- **Where this leads:** the RAG pipeline ([`blueprints/rag-pipeline/`](../../../blueprints/rag-pipeline/)) budgets retrieved context in tokens; the capstone's `llm/` client (Chapter 11) routes token counting through one interface so it follows whichever model you swap in.